In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
from jaxopt import LBFGS
from scipy.optimize import basinhopping, OptimizeResult

# 1. Define a JAX-compatible objective function
def objective_fun(x,a,b):
    # Example: A multi-modal function (Ackley-like)
    return jnp.sum((x-a)**2) + 0.5 * jnp.sum(jnp.cos(2 * jnp.pi * (x-a))) + b

# 2. Define the bridge between SciPy and JAXOpt
# When method is a callable, minimize() calls it with (fun, x0, args=..., **options),
# so we get maxiter/tol/verbose as top-level kwargs, not as options={...}.

def create_lbfgs_minimizer(**kwargs):

    def helper(fun, x0, args=(), **kwargs2):
        """
        A custom minimizer compatible with scipy.optimize.minimize.
        Put maxiter, tol, verbose in minimizer_kwargs["options"]; SciPy unpacks them into kwargs here.
        """
        solver = LBFGS(fun=fun, **kwargs)
        res = solver.run(jnp.array(x0), *args)

        # Pack the results into a SciPy OptimizeResult object
        return OptimizeResult(
            x=np.array(res.params),
            fun=float(res.state.value),
            success=True,  # You can check res.state.error < tol for more rigor
            message="Local minimization finished via JAXOpt LBFGS",
            nit=int(res.state.iter_num)
        )
    
    return helper

kwargs = {
    "maxiter": 2000,
    "tol": 1e-5,
    "verbose": True
}

method = create_lbfgs_minimizer(**kwargs)

# 3. Execute Global Optimization
x0 = np.array([1.0, 1.0])
a,b = 2,5
minimizer_kwargs = {
    "method": method,
    "args": (a, b),
}

result = basinhopping(
    objective_fun, 
    x0, 
    niter=10, 
    minimizer_kwargs=minimizer_kwargs,
)

print(f"Global Minimum found at: {result.x}")
print(f"Objective value: {result.fun}")

INFO: jaxopt.LBFGS: Iter: 1 Gradient Norm (stopping criterion): 0.0 Objective Value:6.0  Stepsize:0.5000001192092896  Number Linesearch Iterations:2 
INFO: jaxopt.LBFGS: Iter: 1 Gradient Norm (stopping criterion): 2.432065725326538 Objective Value:4.658616542816162  Stepsize:0.08368314057588577  Number Linesearch Iterations:3 
INFO: jaxopt.LBFGS: Iter: 2 Gradient Norm (stopping criterion): 2.106599807739258 Objective Value:4.58125638961792  Stepsize:0.12552471458911896  Number Linesearch Iterations:1 
INFO: jaxopt.LBFGS: Iter: 3 Gradient Norm (stopping criterion): 1.6485697031021118 Objective Value:4.525652885437012  Stepsize:0.18828707933425903  Number Linesearch Iterations:1 
INFO: jaxopt.LBFGS: Iter: 4 Gradient Norm (stopping criterion): 1.1285834312438965 Objective Value:4.485710144042969  Stepsize:0.28243061900138855  Number Linesearch Iterations:1 
INFO: jaxopt.LBFGS: Iter: 5 Gradient Norm (stopping criterion): 0.617803156375885 Objective Value:4.462998390197754  Stepsize:0.42364

: 